# CNN-LSTM với Sequence Global Max Pooling

Thí nghiệm này kiểm tra giả thuyết hidden state cuối của LSTM bị ảnh hưởng bởi post-padding. LSTM trả về toàn bộ chuỗi hidden states, sau đó `GlobalMaxPooling1D` chọn tín hiệu ngữ cảnh mạnh nhất.

In [ ]:
from pathlib import Path

print('Working directory:', Path.cwd())
for name in ['train_cnn_lstm_sequence_pool.py', '../../cnn_lstm/CNN_LSTM.py', '../../SQLInjection_XSS_MixDataset.1.0.0.csv', '../../csic_database.csv', '../../obfuscation_dataset_full.xlsx']:
    assert Path(name).exists(), f'Thiếu file: {name}'
print('Đã tìm thấy đầy đủ code và dataset.')

## 1. Xem kiến trúc

In [ ]:
import importlib.util
import sys

path = Path('train_cnn_lstm_sequence_pool.py').resolve()
spec = importlib.util.spec_from_file_location('sequence_pool_experiment', path)
experiment = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = experiment
spec.loader.exec_module(experiment)

demo = experiment.build_sequence_pool_model(vocab_size=191, max_len=1024, embedding_dim=64)
demo.summary()

## 2. Smoke test tùy chọn
Không dùng kết quả smoke test trong báo cáo.

In [ ]:
%run train_cnn_lstm_sequence_pool.py --sample-size 3000 --obfu-sample-size 1000 --epochs 3 --batch-size 128 --output-dir artifacts_sequence_pool_smoke

## 3. Huấn luyện đầy đủ

In [ ]:
%run train_cnn_lstm_sequence_pool.py --epochs 50 --batch-size 128 --output-dir artifacts

## 4. So sánh ba mô hình

In [ ]:
import json
import pandas as pd

def load_json(path):
    with open(path, encoding='utf-8') as f:
        return json.load(f)

cnn = load_json('../../cnn_only/artifacts/metadata_and_results.json')
hybrid = load_json('../../cnn_lstm/artifacts/metadata_and_results.json')
sequence = load_json('artifacts/metadata_and_results.json')

rows = []
for name, data, test_key, attack_key, model_key in [
    ('CNN-only', cnn, 'normal_test', 'Attack (1)', 'cnn_only_model'),
    ('CNN-LSTM final state', hybrid, 'test', 'Attack (1)', 'model'),
    ('CNN-LSTM sequence pool', sequence, 'normal_test', 'Attack (1)', 'sequence_pool_model'),
]:
    test_result = data['evaluation'][test_key]
    obfu_result = data['evaluation']['obfuscated_test']
    model_info = data[model_key]
    rows.append({
        'Mô hình': name,
        'Tham số': model_info.get('parameter_count'),
        'Test Accuracy': test_result['accuracy'],
        'Test AUC': test_result['auc_roc'],
        'Attack Recall': test_result['classification_report'][attack_key]['recall'],
        'Attack F1': test_result['classification_report'][attack_key]['f1-score'],
        'Obfuscated Recall': obfu_result['classification_report']['1']['recall'],
        'Obfuscated Missed': obfu_result['confusion_matrix'][1][0],
    })

pd.DataFrame(rows)

## Cách đọc kết quả

Nếu sequence-pool cải thiện rõ obfuscated recall so với final-state LSTM, giả thuyết padding làm suy giảm hidden state cuối được ủng hộ. Nếu không cải thiện, CNN-only mạnh chủ yếu do đặc trưng cục bộ phù hợp với dữ liệu.